# 21.12 大规模分布式训练 / Distributed Training at Scale

**中文**:训练现代深度学习模型(尤其大语言模型)时,**一块 GPU 根本不够**:要么**数据太多**(训练一遍要几个月)、要么**模型太大**(几百亿参数装不进单卡显存)。解法是把训练**分布到几十、几百、上千块 GPU 上并行**。这是训练 GPT 这类模型的工程核心,也是 ML 系统面试的高频深水区。本节讲清分布式训练的**四种并行策略**(数据/张量/流水线/ZeRO),并从零实现整个体系最核心的通信原语——**Ring-AllReduce**(所有数据并行框架 PyTorch DDP / Horovod 的心脏),亲手验证它为什么能**扩展到上千 GPU 而通信量不爆炸**。
**English**: Training modern deep-learning models (especially large language models), **one GPU is nowhere near enough**: either **too much data** (one pass takes months) or **too big a model** (tens of billions of parameters don't fit in one card's memory). The solution is to **distribute training across tens, hundreds, or thousands of GPUs in parallel**. This is the engineering core of training models like GPT and a frequent deep-end ML-systems interview topic. This section explains distributed training's **four parallelism strategies** (data/tensor/pipeline/ZeRO) and implements from scratch the system's most core communication primitive — **Ring-AllReduce** (the heart of all data-parallel frameworks: PyTorch DDP / Horovod) — verifying by hand why it **scales to thousands of GPUs without the communication blowing up**.

---

**中文**:**四种并行策略(面试要能分清各解决什么问题)**:
**English**: **Four parallelism strategies (know what problem each solves)**:
- **中文**:**数据并行(data parallelism)**:**每块 GPU 有完整模型副本**,各自处理一小批不同数据、算各自的梯度,然后**用 AllReduce 把所有 GPU 的梯度平均**、同步更新。解决"数据太多/想用大 batch 加速"。最常用(PyTorch DDP)。
  **Data parallelism**: **each GPU has a full model replica**, processes a different mini-batch, computes its own gradients, then **averages all GPUs' gradients via AllReduce** and updates in sync. Solves "too much data / want big batches to speed up." Most common (PyTorch DDP).
- **中文**:**张量并行(tensor parallelism)**:**把单个层(如巨大的矩阵乘)横切,分到多块 GPU** 上算,再拼起来。解决"单层大到一块 GPU 都装不下"。GPU 间通信极频繁,需高速互联(NVLink)。
  **Tensor parallelism**: **split a single layer (e.g. a huge matmul) across GPUs**, then stitch results. Solves "one layer too big for a single GPU." Very frequent inter-GPU communication, needs fast interconnect (NVLink).
- **中文**:**流水线并行(pipeline parallelism)**:**把模型按层切成几段,每段放一块 GPU**,数据像流水线一样流过。解决"模型总层数太多、单卡装不下"。用 micro-batch 填满流水线减少 GPU 空闲(bubble)。
  **Pipeline parallelism**: **split the model into stages by layers, one stage per GPU**, data flows through like an assembly line. Solves "too many total layers for one card." Uses micro-batches to fill the pipeline and reduce GPU idle (bubbles).
- **中文**:**ZeRO / FSDP(分片数据并行)**:数据并行虽好,但**每块 GPU 存一份完整的模型+梯度+优化器状态**,内存浪费巨大。ZeRO 把这三者**分片(shard)到各 GPU**,用时才聚合——大幅省显存,让超大模型能训。是训练 LLM 的关键(DeepSpeed ZeRO、PyTorch FSDP)。
  **ZeRO / FSDP (sharded data parallelism)**: data parallelism is nice, but **each GPU stores a full copy of model + gradients + optimizer states**, wasting huge memory. ZeRO **shards** these three across GPUs, gathering only when needed — drastically saving memory so huge models can train. Key to training LLMs (DeepSpeed ZeRO, PyTorch FSDP).

**中文**:**AllReduce 是数据并行的心脏**。数据并行每一步都要"把 N 块 GPU 的梯度加起来,再发回每块 GPU"。朴素做法(所有 GPU 把梯度发给一个 master 求和再广播)会让 master 成为**带宽瓶颈**(收 N 份数据)。**Ring-AllReduce** 是天才的解法:把 GPU 排成环,梯度切成 N 块,通过两个阶段(**scatter-reduce + all-gather**)在环上传递——**每块 GPU 收发的数据量只有 $2(N-1)/N \times D$,几乎与 GPU 数量 N 无关**!所以能扩展到上千卡而不撞带宽墙。
**English**: **AllReduce is the heart of data parallelism.** Every data-parallel step must "sum N GPUs' gradients and send back to each GPU." The naive approach (all GPUs send to one master to sum then broadcast) makes the master a **bandwidth bottleneck** (receiving N copies). **Ring-AllReduce** is the brilliant fix: arrange GPUs in a ring, split gradients into N chunks, and pass them around the ring in two phases (**scatter-reduce + all-gather**) — **each GPU sends/receives only $2(N-1)/N \times D$ of data, nearly independent of the GPU count N**! So it scales to thousands of cards without hitting the bandwidth wall.

> 💡 **面试速查 / Interview cheat-sheet（★★★ ML 系统/大模型必考）**
> **中文**:**四种并行**:①**数据并行**(每 GPU 全模型副本+处理不同数据+**AllReduce 平均梯度**, 解决数据多, DDP 最常用)②**张量并行**(切单层矩阵到多 GPU, 解决单层太大, 通信频繁需 NVLink)③**流水线并行**(按层分段到多 GPU, 解决层多, micro-batch 减 bubble)④**ZeRO/FSDP**(分片模型/梯度/优化器状态省显存, 训 LLM 关键)。大模型常**组合用**(3D 并行=数据×张量×流水线, 如 Megatron)。**AllReduce**=数据并行核心通信:把各 GPU 梯度求和发回; **Ring-AllReduce** 每 GPU 通信量 2(N-1)/N·D **与 N 无关**→可扩展到千卡(vs 朴素 master 汇聚成瓶颈)。**其他关键**:混合精度(fp16/bf16 省显存加速)、梯度累积(小显存模拟大 batch)、梯度检查点(用算力换显存)、通信-计算重叠。**框架**:PyTorch **DDP**(数据并行)/**FSDP**(分片)、**DeepSpeed**(ZeRO)、**Megatron-LM**(张量/流水线)、**Horovod**。**瓶颈永远是通信**(梯度同步), 优化=减少/重叠通信、梯度压缩。面试金句:*"分布式训练四种并行:数据并行(AllReduce 同步梯度, 解决数据多)、张量并行(切层, 解决单层大)、流水线并行(切层段, 解决层多)、ZeRO/FSDP(分片状态省显存); 大模型组合成 3D 并行; 核心通信原语 Ring-AllReduce 每卡通信量与卡数无关故可扩展千卡; 瓶颈是通信, 用混合精度/梯度累积/通信重叠优化。"*
> **English**: **Four parallelisms**: ① **data parallelism** (each GPU a full model replica + different data + **AllReduce averages gradients**, solves too-much-data, DDP most common) ② **tensor parallelism** (split a single layer's matmul across GPUs, solves one-layer-too-big, frequent comm, needs NVLink) ③ **pipeline parallelism** (stages by layers across GPUs, solves too-many-layers, micro-batches reduce bubbles) ④ **ZeRO/FSDP** (shard model/gradients/optimizer states to save memory, key for LLMs). Big models often **combine** them (3D parallelism = data × tensor × pipeline, e.g. Megatron). **AllReduce** = data parallelism's core communication: sum each GPU's gradients and send back; **Ring-AllReduce**'s per-GPU communication 2(N-1)/N·D is **independent of N** → scales to thousands of cards (vs a naive master becoming a bottleneck). **Other keys**: mixed precision (fp16/bf16 saves memory & speeds up), gradient accumulation (simulate big batches on small memory), gradient checkpointing (trade compute for memory), computation-communication overlap. **Frameworks**: PyTorch **DDP** (data parallel) / **FSDP** (sharded), **DeepSpeed** (ZeRO), **Megatron-LM** (tensor/pipeline), **Horovod**. **The bottleneck is always communication** (gradient sync); optimize by reducing/overlapping communication and gradient compression. Interview line: *"Distributed training has four parallelisms: data (AllReduce syncs gradients, solves too-much-data), tensor (split layers, solves one-layer-too-big), pipeline (stage layers, solves too-many-layers), ZeRO/FSDP (shard states to save memory); big models combine into 3D parallelism; the core primitive Ring-AllReduce has per-card communication independent of card count, so it scales to thousands; the bottleneck is communication, optimized with mixed precision/gradient accumulation/comm overlap."*


In [ ]:

# ============================================================
# 从零实现 Ring-AllReduce:数据并行的核心通信原语 / Ring-AllReduce from scratch
# 中文:N 个"GPU"每个有一份梯度向量。目标:每个 GPU 最终都拿到所有梯度的和。用环形通信, 两阶段完成:
#      ① scatter-reduce(N-1步): 每个 GPU 累积出一个完整规约的分块; ② all-gather(N-1步): 把各分块传遍全环。
# English: N "GPUs" each hold a gradient vector. Goal: every GPU ends with the SUM of all gradients, via ring
#      communication in two phases: ① scatter-reduce (N-1 steps): each GPU accumulates one fully-reduced chunk;
#      ② all-gather (N-1 steps): circulate chunks so everyone has all.
# ============================================================
import numpy as np
np.random.seed(0)
N, D = 4, 12
grads=[np.random.randint(1,9,D).astype(float) for _ in range(N)]     # 每个 GPU 的本地梯度 / each GPU's local gradient
true_sum=sum(grads)                                                   # 我们要的正确答案 / the target

def ring_allreduce(vectors):
    N=len(vectors); D=len(vectors[0]); idx=np.array_split(np.arange(D), N)   # 把向量切成 N 块 / split into N chunks
    buf=[v.copy() for v in vectors]; comm=0
    # --- ① scatter-reduce:N-1 步后, GPU w 持有第 (w+1)%N 块的完整和 / after N-1 steps, GPU holds one full-sum chunk ---
    for step in range(N-1):
        new=[b.copy() for b in buf]
        for w in range(N):
            c=(w-step) % N                                            # w 把第 c 块发给右邻居 / w sends chunk c to its right
            new[(w+1)%N][idx[c]] = buf[(w+1)%N][idx[c]] + buf[w][idx[c]]   # 邻居把收到的加到自己对应块 / neighbor adds it
            comm += len(idx[c])                                       # 记通信量 / count communication
        buf=new
    # --- ② all-gather:N-1 步把已规约的分块沿环复制给所有 GPU / circulate reduced chunks to all ---
    for step in range(N-1):
        new=[b.copy() for b in buf]
        for w in range(N):
            c=(w+1-step) % N
            new[(w+1)%N][idx[c]] = buf[w][idx[c]]                     # 覆盖式传递(不再相加)/ copy (not add)
            comm += len(idx[c])
        buf=new
    return buf, comm

result, comm = ring_allreduce(grads)
print("所有 GPU 是否都得到梯度和 / all GPUs converged to the sum:",
      all(np.array_equal(r, true_sum) for r in result))
print("GPU0 结果 / GPU0:", result[0].astype(int))
print("正确和 / true sum:", true_sum.astype(int))
# 通信量对比:ring vs 朴素 master 汇聚 / communication: ring vs naive master-gather
per_worker_ring=comm/N; naive_master=(N-1)*D
print(f"\n每 GPU 通信量 — Ring: {per_worker_ring:.0f} 元素  |  朴素 master 汇聚: master 收 {naive_master} 元素(瓶颈)")
print(f"Ring 每卡通信 = 2·(N-1)/N·D = {2*(N-1)/N*D:.0f}, 几乎与 GPU 数 N 无关 → 可扩展到上千卡!")


In [ ]:

# ============================================================
# 验证 Ring-AllReduce 的可扩展性:通信量不随 GPU 数爆炸 / verify scalability
# 中文:固定每卡梯度大小 D, 变化 GPU 数 N。朴素 master 汇聚:master 收 (N-1)·D(随 N 线性增长, 撞带宽墙);
#      Ring:每卡通信 2(N-1)/N·D → 趋于 2D 的常数上界。这就是它能训千卡集群的原因。
# English: fix per-GPU gradient size D, vary GPU count N. Naive master: master receives (N-1)·D (grows linearly →
#      bandwidth wall); Ring: per-GPU 2(N-1)/N·D → bounded near constant 2D. That's why it scales to thousand-GPU clusters.
# ============================================================
import matplotlib.pyplot as plt
D=1000; Ns=[2,4,8,16,32,64,128,256]
ring_per_gpu=[2*(n-1)/n*D for n in Ns]              # Ring 每卡通信量 / ring per-GPU
master_recv=[(n-1)*D for n in Ns]                   # 朴素 master 接收量 / naive master receives
fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].plot(Ns,master_recv,"o-",color="#C44E52",lw=2,label="朴素 master 汇聚(master 接收)")
ax[0].plot(Ns,ring_per_gpu,"o-",color="#55A868",lw=2,label="Ring-AllReduce(每卡)")
ax[0].set_xlabel("GPU 数 N"); ax[0].set_ylabel("通信量(元素)"); ax[0].set_title("可扩展性:Ring 通信量几乎不随 N 增长")
ax[0].legend(); ax[0].set_xscale("log",base=2)
# 并行策略示意 / parallelism taxonomy
ax[1].axis("off"); ax[1].set_title("四种并行策略解决不同问题",fontsize=12,weight="bold")
rows=[("数据并行 Data","每GPU全模型+不同数据, AllReduce同步梯度","数据太多","#4C72B0"),
      ("张量并行 Tensor","切单层矩阵到多GPU","单层太大装不下","#DD8452"),
      ("流水线并行 Pipeline","按层分段, 每段一GPU","层数太多","#55A868"),
      ("ZeRO / FSDP","分片模型+梯度+优化器状态","省显存训超大模型","#9467BD")]
for i,(name,how,why,c) in enumerate(rows):
    y=0.78-i*0.2
    ax[1].add_patch(plt.Rectangle((0.03,y-0.02),0.94,0.16,fc=c,alpha=0.18,transform=ax[1].transAxes))
    ax[1].text(0.06,y+0.09,name,fontsize=10,weight="bold",color=c,transform=ax[1].transAxes)
    ax[1].text(0.06,y+0.03,how,fontsize=7.5,transform=ax[1].transAxes)
    ax[1].text(0.06,y-0.005,f"→ 解决:{why}",fontsize=7.5,style="italic",color="gray",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/big12_viz.png",dpi=80); plt.show()
print("左:朴素汇聚随 N 线性增长(红, 撞带宽墙), Ring 每卡通信趋于常数(绿)→可扩展千卡")
print("右:四种并行各解决一个瓶颈; 训 LLM 常把它们组合成 3D 并行")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **分布式训练的瓶颈永远是通信,而非计算**:GPU 算得飞快,但训练每一步都要**在 GPU 之间同步梯度**——当你有 1000 块 GPU,如何高效地"把 1000 份梯度加起来发回每块卡"就成了生死问题。Ring-AllReduce 的天才之处,我们亲手验证了:朴素的"汇总到一个 master"方案,master 的接收量随 GPU 数**线性爆炸**(1000 卡就是 999 份数据砸向一个点,带宽瞬间打满);而 Ring-AllReduce 让**每块卡的通信量固定在 $2(N-1)/N \times D \approx 2D$,几乎与卡数无关**——这才是能把训练扩展到上千 GPU 的根本原因。理解"通信是瓶颈"这一点,你就抓住了所有分布式训练优化的主线:减少通信、重叠通信与计算、压缩梯度。
2. **四种并行不是竞争,而是解决不同瓶颈、可以叠加**:①数据太多 → **数据并行**(复制模型、分数据、AllReduce);②单个层太大 → **张量并行**(切矩阵);③层数太多 → **流水线并行**(切层段);④显存不够 → **ZeRO/FSDP**(分片状态)。训练一个几千亿参数的 LLM,往往要**把它们组合起来**(所谓 3D 并行:数据×张量×流水线,如 Megatron-Deepspeed):张量并行在一台机器的 8 卡内(靠 NVLink 高速互联),流水线并行跨机器,数据并行再把整个这套复制多份。**选哪种,取决于你到底撞了哪堵墙**(数据墙?单层显存墙?总层数墙?),而不是哪个更时髦。
3. **诚实的现实:99% 的人用不到、也不该手写这些**。①**框架已经封装好了**:PyTorch **DDP** 一行 `model = DDP(model)` 就是数据并行 + Ring-AllReduce(NCCL 后端);**FSDP**/DeepSpeed 一个 config 就是 ZeRO 分片;你几乎永远不需要手写 AllReduce(我们手写是为了**理解**,不是为了用)。②**先穷尽单卡优化再谈分布式**:混合精度(bf16)、梯度累积(小显存模拟大 batch)、梯度检查点(用重算换显存)、更高效的注意力(FlashAttention)——这些单卡技巧常常让你**根本不需要**上分布式,或让分布式规模小得多。③**多卡不是线性加速**:通信开销、数据加载、同步等待意味着 8 卡通常达不到 8 倍速(能到 6-7 倍就不错);盲目加卡可能因通信瓶颈收益递减甚至变慢。④**成本极高**:千卡训练一次可能几十上百万美元,所以真正做大规模训练的是少数大厂/实验室;大多数人是**微调**(用 LoRA 等参数高效方法,单卡就够)。**结论:理解分布式训练的原理(尤其 AllReduce 和四种并行)是 ML 系统面试的硬通货,但实践中要诚实——先榨干单卡、用好框架、只在真正撞墙时才上分布式,并清醒认识到它的通信瓶颈、非线性加速和高昂成本。**

**English**:
1. **The bottleneck of distributed training is always communication, not computation**: GPUs compute blazingly fast, but every training step must **synchronize gradients across GPUs** — with 1000 GPUs, efficiently "summing 1000 gradients and sending back to each card" becomes a matter of life and death. Ring-AllReduce's genius, which we verified by hand: the naive "gather to one master" scheme makes the master's receive volume **explode linearly** with GPU count (1000 cards means 999 copies slamming into one point, instantly saturating bandwidth); whereas Ring-AllReduce fixes **each card's communication at $2(N-1)/N \times D \approx 2D$, nearly independent of card count** — the fundamental reason training scales to thousands of GPUs. Grasp "communication is the bottleneck" and you have the throughline of all distributed-training optimization: reduce communication, overlap communication with computation, compress gradients.
2. **The four parallelisms don't compete but solve different bottlenecks and stack**: ① too much data → **data parallelism** (replicate model, split data, AllReduce); ② one layer too big → **tensor parallelism** (split matrices); ③ too many layers → **pipeline parallelism** (stage layers); ④ not enough memory → **ZeRO/FSDP** (shard states). Training a several-hundred-billion-parameter LLM often **combines them** (so-called 3D parallelism: data × tensor × pipeline, e.g. Megatron-DeepSpeed): tensor parallelism within one machine's 8 cards (via NVLink fast interconnect), pipeline parallelism across machines, and data parallelism replicating the whole setup. **Which to choose depends on which wall you hit** (data wall? single-layer memory wall? total-layers wall?), not which is trendier.
3. **Honest reality: 99% of people don't need to and shouldn't hand-write this**. ① **Frameworks already encapsulate it**: PyTorch **DDP** — one line `model = DDP(model)` is data parallelism + Ring-AllReduce (NCCL backend); **FSDP**/DeepSpeed — one config is ZeRO sharding; you almost never need to hand-write AllReduce (we did to **understand**, not to use). ② **Exhaust single-GPU optimization before distributing**: mixed precision (bf16), gradient accumulation (simulate big batches on small memory), gradient checkpointing (recompute to save memory), efficient attention (FlashAttention) — these single-GPU tricks often mean you **don't need** to distribute at all, or need far fewer GPUs. ③ **More GPUs isn't linear speedup**: communication overhead, data loading, and sync waits mean 8 GPUs usually don't reach 8x (6-7x is good); blindly adding cards can hit diminishing returns or even slow down due to communication bottlenecks. ④ **Extremely costly**: one thousand-GPU training run can cost hundreds of thousands to millions of dollars, so only a few big tech firms/labs do truly large-scale training; most people **fine-tune** (with parameter-efficient methods like LoRA, one GPU suffices). **Conclusion: understanding distributed training's principles (especially AllReduce and the four parallelisms) is hard currency in ML-systems interviews, but be honest in practice — exhaust single-GPU first, use frameworks well, distribute only when truly hitting a wall, and clearly recognize its communication bottleneck, non-linear speedup, and high cost.**

> 💼 **实战视角 / Practical angle**
> **中文**:分布式训练落地:①**默认从 PyTorch DDP 起步**(数据并行, `torchrun` 启动, NCCL 做 AllReduce)——最常用、最简单;②**显存不够上 FSDP/DeepSpeed ZeRO**(分片优化器/梯度/参数, 训大模型必备);③**超大模型(百亿+)**才需张量/流水线并行(Megatron-LM), 组合成 3D 并行;④**先做单卡优化**:混合精度 `bf16`、梯度累积、梯度检查点、`torch.compile`、FlashAttention——常能免去或缩小分布式;⑤**监控**:GPU 利用率(低=通信/数据瓶颈)、通信占比、scaling efficiency(加卡的实际加速比);⑥**大多数业务是微调**——用 LoRA/QLoRA 等参数高效方法, 单卡/少卡即可, 别一上来就想千卡。面试金句:*"分布式训练四种并行解决不同瓶颈:数据并行(AllReduce 同步梯度)、张量并行(切单层)、流水线并行(切层段)、ZeRO/FSDP(分片状态省显存), 大模型组合成 3D 并行; Ring-AllReduce 每卡通信量与卡数无关是可扩展千卡的关键; 瓶颈是通信不是算力, 实践上先榨干单卡(混合精度/梯度累积/检查点)、用 DDP/FSDP 框架、只在真撞墙时才扩规模。"*
> **English**: Distributed training in practice: ① **start with PyTorch DDP by default** (data parallelism, launch with `torchrun`, NCCL for AllReduce) — most common and simplest; ② **out of memory → FSDP/DeepSpeed ZeRO** (shard optimizer/gradients/params, essential for big models); ③ **only very large models (tens of billions+)** need tensor/pipeline parallelism (Megatron-LM), combined into 3D parallelism; ④ **do single-GPU optimization first**: mixed precision `bf16`, gradient accumulation, gradient checkpointing, `torch.compile`, FlashAttention — often removing or shrinking the need to distribute; ⑤ **monitor**: GPU utilization (low = communication/data bottleneck), communication fraction, scaling efficiency (actual speedup per added card); ⑥ **most business work is fine-tuning** — use parameter-efficient methods like LoRA/QLoRA on one/few GPUs; don't reach for thousand-GPU from the start. Interview line: *"Distributed training's four parallelisms solve different bottlenecks: data (AllReduce syncs gradients), tensor (split a layer), pipeline (stage layers), ZeRO/FSDP (shard states to save memory), combined into 3D parallelism for big models; Ring-AllReduce's per-card communication being independent of card count is the key to scaling to thousands; the bottleneck is communication not compute, so in practice exhaust single-GPU first (mixed precision/gradient accumulation/checkpointing), use DDP/FSDP frameworks, and scale only when truly hitting a wall."*

---
### 小结 / Summary
- **中文**:分布式训练四种并行:数据(AllReduce 同步梯度)、张量(切单层)、流水线(切层段)、ZeRO/FSDP(分片状态省显存), 大模型组合成 3D 并行。
- **English**: Four parallelisms: data (AllReduce syncs gradients), tensor (split a layer), pipeline (stage layers), ZeRO/FSDP (shard states); combined into 3D parallelism for big models.
- **中文**:Ring-AllReduce 每卡通信量 2(N-1)/N·D 与卡数无关 → 可扩展千卡(朴素 master 汇聚会成瓶颈)。
- **English**: Ring-AllReduce's per-card communication 2(N-1)/N·D is independent of card count → scales to thousands (a naive master gather bottlenecks).
- **中文**:瓶颈永远是通信不是算力; 实践先榨干单卡(混合精度/梯度累积/检查点)、用 DDP/FSDP、真撞墙才扩规模。
- **English**: The bottleneck is always communication, not compute; in practice exhaust single-GPU first (mixed precision/gradient accumulation/checkpointing), use DDP/FSDP, scale only when truly hitting a wall.
